# Evaluasi Retrieval ChatKUHP dengan RAGAS

Notebook ini mengevaluasi performa sistem retrieval **ChatKUHP** menggunakan framework [RAGAS](https://docs.ragas.io/).

## Metrik yang Digunakan

| Metrik | Penjelasan |
|---|---|
| **Context Precision** | Seberapa presisi konteks yang diambil — apakah node yang diambil memang relevan dengan ground truth? |
| **Context Recall** | Seberapa lengkap konteks yang diambil — apakah semua informasi yang dibutuhkan ada dalam konteks yang diambil? |

> **Catatan:** Evaluasi ini hanya menguji komponen *retrieval* (endpoint `/getcontext`), bukan kualitas jawaban LLM. Endpoint `/getcontext` mengembalikan `selected_hierarchy` berupa pasal-pasal hukum hasil DFS yang dipilih sebagai konteks.

## Dependency

In [ ]:
# Jalankan sekali untuk instalasi
!pip install --upgrade ragas "langchain-community<0.3.0" --upgrade langchain-cohere langchain-google-genai datasets pandas requests

  Using cached langchain_cohere-0.6.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached langchain_google_genai-4.2.6-py3-none-any.whl.metadata (2.7 kB)
INFO: pip is looking at multiple versions of langchain-cohere to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_cohere-0.5.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached langchain_cohere-0.5.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached langchain_cohere-0.4.6-py3-none-any.whl.metadata (6.6 kB)
  Using cached langchain_cohere-0.4.5-py3-none-any.whl.metadata (6.6 kB)
  Using cached langchain_cohere-0.4.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached langchain_cohere-0.4.3-py3-none-any.whl.metadata (6.6 kB)
  Using cached langchain_cohere-0.4.2-py3-none-any.whl.metadata (6.6 kB)
INFO: pip is still looking at multiple versions of langchain-cohere to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_

## Import & Configuration

In [ ]:
import os
import json
import requests
import pandas as pd
from typing import List, Dict
from datasets import Dataset as HFDataset

from ragas import evaluate
from ragas.metrics import context_precision, context_recall

from langchain_cohere import ChatCohere
from langchain_cohere import CohereEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

/tmp/ipykernel_2542/3274036736.py:9: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall
/tmp/ipykernel_2542/3274036736.py:9: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall


## Variable Configuration

In [ ]:
# ── Konfigurasi ──────────────────────────────────────────────────────
try:
    from google.colab import userdata
    COHERE_API_KEY = userdata.get("COHERE_API_KEY") or os.environ.get("COHERE_API_KEY", "YOUR_COHERE_API_KEY")
    URL_SERVER     = userdata.get("URL_SERVER") or "http://localhost:8000"
except ImportError:
    COHERE_API_KEY = os.environ.get("COHERE_API_KEY", "YOUR_COHERE_API_KEY")
    URL_SERVER     = "http://localhost:8000"

CSV_PATH      = "ground_truth.csv"
REQUEST_TIMEOUT = 120

# Download ground_truth.csv jika tidak ditemukan di session storage Colab
if not os.path.exists(CSV_PATH):
    print(f"'{CSV_PATH}' tidak ditemukan. Mengunduh dari GitHub...")
    github_csv_url = "https://raw.githubusercontent.com/Kanaieu/chatKUHP/main/evaluation/ground_truth.csv"
    try:
        import requests
        resp = requests.get(github_csv_url, timeout=10)
        resp.raise_for_status()
        with open(CSV_PATH, "wb") as f:
            f.write(resp.content)
        print("Download berhasil!")
    except Exception as e:
        print(f"Gagal mengunduh dari GitHub: {e}")
        print("Silakan upload file 'ground_truth.csv' secara manual ke session storage Colab.")

'ground_truth.csv' tidak ditemukan. Mengunduh dari GitHub...
Download berhasil!


## Helper Function

### `get_context(url_server, question)`
Memanggil endpoint `/getcontext` pada backend untuk mendapatkan konteks hukum (pasal-pasal hasil DFS) dari sebuah pertanyaan.

Endpoint mengembalikan:
```json
{
  "chosen_goal": "KUHP Pasal 362",
  "goal_choices": [...],
  "contexts": { "KUHP Pasal 362": { "description": "...", ... }, ... }
}
```

RAGAS mengharapkan `contexts` berupa `List[str]`, sehingga kita akan mengekstrak teks deskripsi dari setiap node.

### `load_dataset(csv_path)`
Membaca CSV ground truth. CSV harus memiliki kolom:
- `question` — pertanyaan atau kasus hukum pengguna
- `ground_truth` — jawaban referensi yang benar

### `build_ragas_dataset(df, url_server)`
Membangun dataset dalam format yang dibutuhkan RAGAS dengan memanggil `/getcontext` untuk setiap baris data.

In [ ]:
def get_context(url_server: str, question: str) -> List[str]:
    """
    Memanggil endpoint /getcontext pada backend dan mengembalikan
    list teks konteks hukum (deskripsi pasal hasil DFS).
    """
    url = url_server.rstrip("/") + "/getcontext"
    payload = {"query": question}
    headers = {"Content-Type": "application/json"}

    resp = requests.post(url, headers=headers, json=payload, timeout=REQUEST_TIMEOUT)
    resp.raise_for_status()
    data = resp.json()

    contexts_dict: dict = data.get("contexts", {})
    context_texts: List[str] = []
    for node_name, node_data in contexts_dict.items():
        desc = node_data.get("description", "")
        if desc:
            context_texts.append(f"{node_name}: {desc}")

    if not context_texts:
        chosen = data.get("chosen_goal", "")
        if chosen:
            context_texts = [chosen]

    return context_texts


def load_dataset(csv_path: str) -> pd.DataFrame:
    """
    Membaca CSV ground truth dan memvalidasi kolom yang diperlukan.
    """
    df = pd.read_csv(csv_path)
    if "question" not in df.columns:
        raise ValueError("CSV tidak memiliki kolom 'question'")
    if "ground_truth" not in df.columns:
        raise ValueError("CSV tidak memiliki kolom 'ground_truth'")
    return df


def build_ragas_dataset(df: pd.DataFrame, url_server: str) -> Dict[str, List]:
    """
    Membangun dataset RAGAS dari DataFrame.
    Untuk setiap baris, memanggil /getcontext untuk mendapatkan konteks retrieval.

    Format output yang dibutuhkan RAGAS:
    {
        "question"    : List[str]
        "answer"      : List[str]   -- kosong karena hanya eval retrieval
        "contexts"    : List[List[str]]
        "ground_truth": List[str]
    }
    """
    questions:    List[str]       = []
    answers:      List[str]       = []
    contexts:     List[List[str]] = []
    ground_truths: List[str]      = []

    total = len(df)
    for i, (_, row) in enumerate(df.iterrows()):
        q  = str(row["question"]).strip()
        gt = str(row.get("ground_truth", "")).strip()

        if not q:
            print(f"[SKIP] Baris {i+1}: pertanyaan kosong.")
            continue

        print(f"[{i+1}/{total}] Mengambil konteks untuk: {q[:80]}...")
        try:
            ctx = get_context(url_server, q)
        except Exception as e:
            print(f"  → ERROR: {e}. Konteks dikosongkan.")
            ctx = []

        questions.append(q)
        answers.append("")     # dikosongkan — hanya evaluasi retrieval
        contexts.append(ctx)
        ground_truths.append(gt)

    return {
        "question":    questions,
        "answer":      answers,
        "contexts":    contexts,
        "ground_truth": ground_truths,
    }

## Load Dataset & Build RAGAS Dataset

In [ ]:
df = load_dataset(CSV_PATH)
print(f"Total data ground truth: {len(df)} baris")
df.head()

Total data ground truth: 20 baris


,question,ground_truth
0,Bisakah seseorang yang menghasut terjadinya ti...,Tindak pidana penghasutan diatur dalam Pasal 2...
1,Apa hukuman untuk seseorang yang menggunakan n...,Ketentuan mengenai larangan menyebutkan identi...
2,Saya membeli emas 2 gram berbentuk cincin yang...,Tindak pidana penipuan diatur dalam Pasal 492 ...
3,"Suatu ketika, saya memergoki istri saya sedang...",Tindak pidana perzinaan diatur dalam Pasal 411...
4,Ada kasus pencurian listrik oleh tetangga dan ...,Ketentuan mengenai tindak pidana pencurian dia...


In [ ]:
print("Membangun RAGAS dataset (memanggil /getcontext untuk setiap pertanyaan)...")
ragas_dict = build_ragas_dataset(df, URL_SERVER)

hf_ragas = HFDataset.from_dict(ragas_dict)
print(f"\nDataset berhasil dibuat: {hf_ragas}")

Membangun RAGAS dataset (memanggil /getcontext untuk setiap pertanyaan)...
[1/20] Mengambil konteks untuk: Bisakah seseorang yang menghasut terjadinya tindak pidana dijerat dengan KUHP pe...
[2/20] Mengambil konteks untuk: Apa hukuman untuk seseorang yang menggunakan nama palsu?...
[3/20] Mengambil konteks untuk: Saya membeli emas 2 gram berbentuk cincin yang sudah ada logo beratnya. Selesai ...
[4/20] Mengambil konteks untuk: Suatu ketika, saya memergoki istri saya sedang berselingkuh hingga melakukan hub...
[5/20] Mengambil konteks untuk: Ada kasus pencurian listrik oleh tetangga dan dikenakan denda pencurian listrik....
[6/20] Mengambil konteks untuk: Dalam UU 1/2023 tentang KUHP baru, tindak pidana penodaan terhadap Bendera Negar...
[7/20] Mengambil konteks untuk: Apakah pengaduan kohabitasi bisa dicabut, kapan batas waktunya?...
[8/20] Mengambil konteks untuk: Apakah aksi pelemparan batu ke kereta api bisa dipidana? Jika bisa, apa sanksi h...
[9/20] Mengambil konteks untuk: Apakah

## Setup LLM Judge for RAGAS

In [ ]:
# LLM judge menggunakan Cohere
llm_judge = LangchainLLMWrapper(
    ChatCohere(
        model="command-a-03-2025",
        cohere_api_key=COHERE_API_KEY,
        temperature=0.0,
    )
)

emb_judge = LangchainEmbeddingsWrapper(
    CohereEmbeddings(
        model="embed-multilingual-v3.0",
        cohere_api_key=COHERE_API_KEY,
    )
)

/tmp/ipykernel_2542/3133088878.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  llm_judge = LangchainLLMWrapper(
/tmp/ipykernel_2542/3133088878.py:10: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  emb_judge = LangchainEmbeddingsWrapper(


## RAGAS Evaluation

In [ ]:
print("Menjalankan evaluasi RAGAS...")

result = evaluate(
    dataset=hf_ragas,
    metrics=[context_precision, context_recall],
    llm=llm_judge,
    embeddings=emb_judge,
)

print("\n===== HASIL EVALUASI RAGAS =====")
print(result)

Menjalankan evaluasi RAGAS...


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]


===== HASIL EVALUASI RAGAS =====
{'context_precision': 0.8250, 'context_recall': 0.7500}


## Result as Dataframe

In [ ]:
result_df = result.to_pandas()
result_df["rewritten_query"] = hf_ragas["rewritten_query"]
print("\nRingkasan Metrik:")
print(f"  Context Precision : {result_df['context_precision'].mean():.4f}")
print(f"  Context Recall    : {result_df['context_recall'].mean():.4f}")

result_df


Ringkasan Metrik:
  Context Precision : 0.8250
  Context Recall    : 0.7500


,user_input,retrieved_contexts,response,reference,context_precision,context_recall,rewritten_query
0,Bisakah seseorang yang menghasut terjadinya ti...,[KUHP Pasal 241 Ayat 2: Dalam hal Tindak Pidan...,,Tindak pidana penghasutan diatur dalam Pasal 2...,0.0,0.0,"Penghasutan tindak pidana, mengajak melakukan ..."
1,Apa hukuman untuk seseorang yang menggunakan n...,[KUHP Pasal 292 Ayat 2: Ketentuan sebagaimana ...,,Ketentuan mengenai larangan menyebutkan identi...,0.5,1.0,"Penggunaan identitas palsu, menggunakan nama p..."
2,Saya membeli emas 2 gram berbentuk cincin yang...,[KUHP Pasal 494: Dipidana karena penipuan ring...,,Tindak pidana penipuan diatur dalam Pasal 492 ...,0.5,1.0,"Penipuan jual beli barang, berat emas tidak se..."
3,"Suatu ketika, saya memergoki istri saya sedang...",[KUHP Pasal 411 Ayat 2: Terhadap Tindak Pidana...,,Tindak pidana perzinaan diatur dalam Pasal 411...,1.0,1.0,"Perzinahan, hubungan badan dengan pasangan ora..."
4,Ada kasus pencurian listrik oleh tetangga dan ...,[KUHP Pasal 478: Jika Tindak Pidana sebagaiman...,,Ketentuan mengenai tindak pidana pencurian dia...,0.5,1.0,"Pencurian listrik, mengambil aliran listrik se..."
5,"Dalam UU 1/2023 tentang KUHP baru, tindak pida...","[KUHP Pasal 236: Semua Orang yang mencoret, me...",,Tindak pidana penodaan terhadap lambang negara...,1.0,0.5,"Penodaan Bendera Negara dan Lambang Negara, me..."
6,"Apakah pengaduan kohabitasi bisa dicabut, kapa...",[KUHP Pasal 30 Ayat 1: Pengaduan dapat ditarik...,,Tindak pidana kohabitasi atau hidup bersama se...,1.0,0.0,"Kohabitasi, pencabutan pengaduan kohabitasi, b..."
7,Apakah aksi pelemparan batu ke kereta api bisa...,[KUHP Pasal 324 Ayat 1: Setiap Orang yang kare...,,Perbuatan yang karena kealpaannya mengakibatka...,1.0,1.0,"Kealpaan bahaya lalu lintas kereta api, melemp..."
8,Apakah sanksi hukum bagi warga negara Indonesi...,[KUHP Pasal 261 Ayat 2: Pendiri atau pengurus ...,,Ketentuan mengenai larangan bergabung dalam or...,1.0,1.0,"Makr, membentuk perkumpulan bukan badan hukum ..."
9,Bagaimana jerat hukum menghalangi kegiatan iba...,[KUHP Pasal 274 Ayat 2: Semua Orang yang melak...,,Ketentuan mengenai penyelenggaraan pesta atau ...,1.0,1.0,"Gangguan ketertiban kegiatan keagamaan, mengha..."


In [ ]:
# Simpan hasil ke CSV (opsional)
output_path = "ragas_results.csv"
result_df.to_csv(output_path, index=False)
print(f"Hasil disimpan ke: {output_path}")

Hasil disimpan ke: ragas_results.csv
